# Document processing: report-level sentiment

Runs the single extraction script over the 10-K and 10-Q samples in `input_data/`
and writes report-level polarities to `sentiment_extraction_results/`.

All the logic lives in `code/scripts/`; this notebook only configures and runs it.

One procedure for both form types: the filing is split into token-budgeted chunks,
each chunk gets the prompt ending appended, one token position is read out, the
top-p nucleus is matched against the verbalizer, and the chunk scores are averaged
into one row per filing.

Each filing is keyed by `(CIK, FILING_DATE, ACC_NUM, FORM_TYPE)`, so a re-run
resumes exactly where it stopped and merges stay safe.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]  # code/document processing -> code -> repo root
sys.path.insert(0, str(REPO_ROOT / "code" / "scripts"))

import pandas as pd

from utils import INPUT_DATA_DIR, SENTIMENT_RESULTS_DIR, clean_mem, set_seed
from reports import discover_reports
from prompts import STRATEGIES
from sentiment_extraction import ExtractionConfig, ScoringBackend, run_extraction

pd.set_option("display.max_columns", None)
set_seed(42)

print("repo:", REPO_ROOT)
print("input:", INPUT_DATA_DIR)
print("output:", SENTIMENT_RESULTS_DIR)
print("strategies:", sorted(STRATEGIES))

## Sample

File names carry the filing key, so the sample is just whatever sits in
`input_data/`. Defaults per form type: Item 7 (MD&A) for a 10-K, the whole
document for a 10-Q.

In [ ]:
sample = discover_reports(INPUT_DATA_DIR)

print(sample["FORM_TYPE"].value_counts().to_string())
print()
print("filing years:", sorted(sample["FILING_DATE"].str[:4].unique()))
sample.head()

## Configuration

`chunk_size_tokens=None` keeps the earlier convention: chunk size is the model
window minus a reserve for the prompt ending (1000 tokens for a causal LM, 100
for a masked LM). Note what that implies per model:

- **ModernBERT-large** (window 8192) genuinely chunks long filings.
- **Llama-3.1-8B** (window 131072) leaves ~130k tokens per chunk, so almost
  every filing is a single chunk and the averaging step is a no-op.

Set `chunk_size_tokens` explicitly (e.g. 4096) if the two models should see the
same amount of text per readout.

In [ ]:
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"  # or a local path
STRATEGY = "growth"                        # "growth" | "financial_risk"

common = dict(
    model_id=MODEL_ID,
    strategy=STRATEGY,
    input_dir=INPUT_DATA_DIR,
    output_dir=SENTIMENT_RESULTS_DIR,
    chunk_overlap_tokens=240,
    top_p=0.9,
    docs_per_batch=2,
    save_every=200,
    resume=True,
)

cfg_10q = ExtractionConfig(form_type="10-Q", **common)
cfg_10k = ExtractionConfig(form_type="10-K", **common)

for cfg in (cfg_10q, cfg_10k):
    print(cfg.form_type, "->", cfg.resolve_output_path().name)

## Smoke test

Score a handful of filings first. Watch the log lines for the resolved chunk
size and for any verbalizer words that have no single-token form -- those
contribute nothing to the score.

In [ ]:
from dataclasses import replace

smoke = replace(cfg_10q, limit=5, output_name=f"_smoke_{STRATEGY}_10Q.csv", resume=False)
smoke_result = run_extraction(smoke)
smoke_result

## Inspect the readout

What the model actually puts at the readout position for one filing -- the
starting point for any prompt or verbalizer change. Loads its own copy of the
model, so free it before the full run.

In [ ]:
from reports import ReportDataset
from prompts import normalize_token

backend = ScoringBackend(cfg_10q)
strategy = STRATEGIES[STRATEGY]

# first filing that is actually long enough to score
item = next(i for i in ReportDataset(sample[sample["FORM_TYPE"] == "10-Q"]) if i.ok)
prompt = strategy.build_prompt(item.text[:20_000], backend.kind, backend.mask_token)

print(item.key, f"({item.n_words} words)")
print("ending:", strategy.ending(backend.kind, backend.mask_token))
print()
for token, prob in backend.top_tokens(prompt, k=20):
    word = normalize_token(token)
    side = (
        "positive" if word in strategy.verbalizer["positive"]
        else "negative" if word in strategy.verbalizer["negative"]
        else ""
    )
    print(f"  {prob:7.4f}  {token!r:20s} {side}")

In [ ]:
del backend
clean_mem()

## Full run

Safe to interrupt and re-run: finished filings are skipped on resume, and
anything that could not be scored lands in the matching `*.failures.csv`.

In [ ]:
res_10q = run_extraction(cfg_10q)
res_10q.shape

In [ ]:
res_10k = run_extraction(cfg_10k)
res_10k.shape

## Results overview

`matched_mass` is how much of the nucleus the verbalizer captured -- a coverage
diagnostic. Low values mean the model mostly answered with words outside the
word lists, and the polarity of those filings rests on little probability mass.

`polarity_norm = polarity / matched_mass` separates direction from that mass;
`polarity_chunk_std` shows how much the chunks of one filing disagreed.

In [ ]:
scores = pd.concat([res_10q, res_10k], ignore_index=True)

cols = ["polarity", "polarity_norm", "matched_mass", "n_chunks", "n_words"]
scores.groupby("FORM_TYPE")[cols].describe().T.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for form, group in scores.groupby("FORM_TYPE"):
    axes[0].hist(group["polarity"], bins=60, histtype="step", density=True, label=form)
    axes[1].hist(group["polarity_norm"].dropna(), bins=60, histtype="step", density=True, label=form)
    axes[2].hist(group["matched_mass"], bins=60, histtype="step", density=True, label=form)

for ax, title in zip(axes, ["polarity", "polarity_norm", "matched_mass"]):
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Anything that could not be scored, with the reason.
for cfg in (cfg_10q, cfg_10k):
    path = cfg.resolve_output_path()
    failures = path.with_name(path.stem + ".failures.csv")
    if failures.exists():
        frame = pd.read_csv(failures)
        print(f"{cfg.form_type}: {len(frame)} failure(s)")
        print(frame["reason"].str.split(":").str[0].value_counts().to_string())
    else:
        print(f"{cfg.form_type}: no failures")